In [93]:
spark

In [94]:
%run /opt/spark-notebooks/silver_base/parsed_data.ipynb


Master DataFrame 'parsed_df' successfully created and listening to Bronze!


In [95]:
from pyspark.sql.functions import col, explode
# 1. Create the database table mapping
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_base.order_item_fact_RT (
        order_id STRING,
        item_id STRING,
        product_id STRING,
        price DECIMAL(10,2),
        quantity INT,
        source_system STRING,
        bronze_ingest_ts TIMESTAMP
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/base/order_item_fact_RT'
""")

DataFrame[]

In [96]:
items_exploded_df = parsed_df.select(
    col("data.order.order_id").alias("order_id"),
    explode(col("data.order.items")).alias("item"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [97]:
silver_order_item_fact_df = items_exploded_df.select(
    col("order_id"),
    col("item.item_id").alias("item_id"),
    col("item.product_details.product_id").alias("product_id"),
    col("item.price").cast("decimal(10,2)").alias("price"),
    col("item.quantity").cast("int").alias("quantity"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [98]:
item_fact_query = (
    silver_order_item_fact_df.writeStream
    .format("delta")
    .queryName("silver_order_item_fact_stream")
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark-data/checkpoints/silver/base/order_item_fact_RT/v1")
    .trigger(processingTime="30 seconds")
    .start("/opt/spark-data/delta/silver/base/order_item_fact_RT")
)
print("Started silver_order_item_fact_RT continuous stream! 🛒")

Started silver_order_item_fact_RT continuous stream! 🛒


In [100]:
spark.sql("SELECT * FROM silver_base.order_item_fact_RT").limit(20).toPandas()


,order_id,item_id,product_id,price,quantity,source_system,bronze_ingest_ts
0,ORD-10001,ITEM-10001,P-101,55000.00,1,order_service,2026-03-27 16:27:10.891
1,ORD-10002,ITEM-10002,P-202,18000.00,1,order_service,2026-03-27 16:27:10.891
2,ORD-10003,ITEM-10003,P-303,6500.00,2,order_service,2026-03-27 16:27:10.891
3,ORD-10004,ITEM-10004,P-404,9000.00,1,order_service,2026-03-27 16:27:10.891
4,ORD-10005,ITEM-10005,P-505,42000.00,1,order_service,2026-03-27 16:27:10.891


In [320]:
spark.sql("SELECT * FROM silver_base.order_item_fact_RT").limit(20).toPandas()


,order_id,item_id,product_id,price,quantity,source_system,bronze_ingest_ts
0,ORD-10001,ITEM-10001,P-101,55000.00,1,order_service,2026-03-27 07:29:52.410
1,ORD-10002,ITEM-10002,P-202,18000.00,1,order_service,2026-03-27 07:29:52.410
2,ORD-10003,ITEM-10003,P-303,6500.00,2,order_service,2026-03-27 07:29:52.410
3,ORD-10004,ITEM-10004,P-404,9000.00,1,order_service,2026-03-27 07:29:52.410
4,ORD-10005,ITEM-10005,P-505,42000.00,1,order_service,2026-03-27 07:29:52.410
